# Text-Only Linear Regression (384 embedding dimensions)

Same design as `prediction_linear_regression_all_features.ipynb` -- monthly refit, 252-trading-day
rolling window, OLS, daily out-of-sample predictions -- but the **only regressors are the 384
mean-pooled sentence-embedding dimensions** of the stock-day's messages. No sentiment, volume or
attention features, and no message count (`embed_n` is attention, not content).

Input is `Data/text_master.pkl` (built by `02 - prepare training dataset/build_text_master.ipynb`):
one row per training stock-day that has message text, with the panel's row label in `mm_index`.
Predictions therefore exist only for stock-days with messages; the prediction file's `index`
column is the `merged_master` row label, as in every other prediction file.

Output: `Data/predictions_linear_regression_textonly_input=384.pkl`. The distinct model name keeps
this file invisible to the `find_all_features_file()` resolvers in 04/05, which pick the largest
`input=N` among `predictions_linear_regression_input=*.pkl`; it must be registered explicitly.


In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from joblib import Parallel, delayed


In [ ]:
# Directories
CODE_DIR = Path(r"D:\StockTwits\Code")
DATA_DIR = Path(r"D:\StockTwits\Data\v1\data\csv")
FIGURES_DIR = Path(r"D:\StockTwits\Figures")
MODEL_DATA_DIR = Path(r"D:\StockTwits\Data")

# File names
INPUT_DATA = MODEL_DATA_DIR / "text_master.pkl"
MODEL_NAME = "linear_regression_textonly"

# Number of threads for parallel processing (each month's design matrix is up to ~600k x 384 float64)
N_JOBS = 4


In [ ]:
# Load data (row label = merged_master row label, so predictions can be placed back onto the panel)
data = pd.read_pickle(INPUT_DATA).set_index("mm_index")
data["date"] = pd.to_datetime(data["date"])
print(f"Loaded {len(data):,} stock-days with text, {data['date'].min().date()} to {data['date'].max().date()}")


In [ ]:
# Prepare features and target
TARGET = 'f_cumret1'

# Text only: the 384 embedding dimensions. embed_n (message count) is deliberately excluded.
FEATURES = [c for c in data.columns if c.startswith('embed_') and c != 'embed_n']
assert len(FEATURES) == 384, len(FEATURES)

# Remove missing values (only the target can be missing here)
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Number of features: {len(FEATURES)}")


In [ ]:
# Add date column and sort
model_data['date'] = data.loc[model_data.index, 'date']
model_data = model_data.sort_values('date')
model_data['year_month'] = model_data['date'].dt.to_period('M')


# In-Sample linear regression

In [ ]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit linear regression model (in-sample)
lr_model = LinearRegression()
lr_model.fit(X, y)

# Make predictions
y_pred = lr_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Linear Regression Results (text only)")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print(f"Intercept: {lr_model.intercept_:.6f}")
print(f"Coefficient magnitude: median |b| = {np.median(np.abs(lr_model.coef_)):.5f}, max |b| = {np.abs(lr_model.coef_).max():.5f}")


# OOS Predictions (Monthly Training, Daily Predictions)

In [ ]:
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW = 252  # Rolling window size in trading days

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

# Get unique months in the OOS period
oos_months = model_data.loc[model_data['date'] > train_end, 'year_month'].unique()
oos_months = sorted(oos_months)

print(f"Rolling window: {WINDOW} trading days")
print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")
print(f"Number of OOS months: {len(oos_months)}")
print(f"Parallel processing: {N_JOBS} threads")


In [ ]:
def train_and_predict_month(month_idx, pred_month, model_data, unique_dates, oos_dates, FEATURES, TARGET, WINDOW):
    """
    Train linear regression on rolling window and predict for all days in a month.
    Designed for parallel execution with joblib.
    """
    # Get all prediction dates in this month
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]

    if len(month_dates) == 0:
        return []

    # Training cutoff: end of the previous month
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)

    # Find the last trading day before or on train_cutoff
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return []
    last_train_date = train_dates[-1]

    # ROLLING WINDOW: Train on last WINDOW trading days before the month
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return []

    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    train_mask = (model_data['date'] >= start_date) & (model_data['date'] <= last_train_date)
    X_train = model_data.loc[train_mask, FEATURES]
    y_train = model_data.loc[train_mask, TARGET]

    if len(X_train) == 0:
        return []

    # Train model
    lr_model = LinearRegression()
    lr_model.fit(X_train, y_train)

    # Predict for all days in the month
    month_predictions = []
    for pred_date in month_dates:
        test_mask = model_data['date'] == pred_date
        X_test = model_data.loc[test_mask, FEATURES]

        if len(X_test) == 0:
            continue

        test_indices = model_data.index[test_mask]
        y_pred = lr_model.predict(X_test)

        for idx, pred in zip(test_indices, y_pred):
            month_predictions.append({
                'date': pred_date,
                'index': idx,
                'prediction': pred
            })

    return month_predictions


In [ ]:
# Run parallel processing across months
print(f"Starting parallel processing with {N_JOBS} jobs...")
print(f"Processing {len(oos_months)} months...")

def get_month_slice(pred_month):
    """Pre-slice model_data to the rows needed for this month's rolling window + predictions,
    so each parallel worker receives a small DataFrame instead of a full copy of model_data."""
    month_dates = oos_dates[oos_dates.to_period('M') == pred_month]
    if len(month_dates) == 0:
        return model_data.iloc[0:0]
    first_day_of_month = pred_month.to_timestamp()
    train_cutoff = first_day_of_month - pd.Timedelta(days=2)
    train_dates = unique_dates[unique_dates <= train_cutoff]
    if len(train_dates) == 0:
        return model_data.iloc[0:0]
    last_train_date = train_dates[-1]
    last_train_date_idx = unique_dates.get_loc(last_train_date)
    if last_train_date_idx < WINDOW:
        return model_data.iloc[0:0]
    start_date = unique_dates[last_train_date_idx - WINDOW + 1]
    end_date = month_dates.max()
    return model_data.loc[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)]

all_results = Parallel(n_jobs=N_JOBS, verbose=10, backend='threading')(
    delayed(train_and_predict_month)(
        month_idx, pred_month, get_month_slice(pred_month), unique_dates, oos_dates,
        FEATURES, TARGET, WINDOW
    )
    for month_idx, pred_month in enumerate(oos_months)
)

# Flatten results
predictions = [pred for month_preds in all_results for pred in month_preds]

print(f"\nCompleted.")
print(f"Total predictions: {len(predictions):,}")


In [ ]:
# Convert predictions to DataFrame
predictions_df = pd.DataFrame(predictions)

# Add symbol information (data is indexed by the merged_master row label)
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index().rename(columns={'mm_index': 'index'}),
    on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'prediction']]

# Sort by date and ticker
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"Non-null predictions: {predictions_df['prediction'].notna().sum():,}")
print(f"Prediction spread: std {predictions_df['prediction'].std():.5f}, "
      f"1st/99th pct {predictions_df['prediction'].quantile(0.01):.5f} / {predictions_df['prediction'].quantile(0.99):.5f}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))


In [ ]:
# Save predictions with dynamic filename
OUTPUT_FILE = MODEL_DATA_DIR / f"predictions_{MODEL_NAME}_input={len(FEATURES)}.pkl"
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {OUTPUT_FILE.stat().st_size / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")
print(f"Columns: {list(predictions_df.columns)}")
